In [1]:
import pandas as pd
import numpy as np
import os
import glob

print("ML dataset preparation started!")

ML dataset preparation started!


In [2]:
processed_path = "../data/CICIDS2017/processed"

processed_files = glob.glob(
    os.path.join(processed_path, "*.csv")
)

print("Files found:", len(processed_files))

for file in processed_files:
    print(os.path.basename(file))

Files found: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [3]:
columns_to_drop = [
    "Flow ID",
    "Source IP",
    "Destination IP",
    "Timestamp",
    "Label"
]

In [4]:
MAX_PER_CLASS_PER_FILE = 25000

all_samples = []

for file in processed_files:
    print("\nProcessing:", os.path.basename(file))
    
    data = pd.read_csv(file)
    
    data.columns = data.columns.str.strip()
    
    # Separate benign and attack traffic
    benign = data[data["Label"] == "BENIGN"]
    attack = data[data["Label"] != "BENIGN"]
    
    # Sample from each class
    benign_sample = benign.sample(
        n=min(MAX_PER_CLASS_PER_FILE, len(benign)),
        random_state=42
    )
    
    attack_sample = attack.sample(
        n=min(MAX_PER_CLASS_PER_FILE, len(attack)),
        random_state=42
    )
    
    file_sample = pd.concat([
        benign_sample,
        attack_sample
    ])
    
    all_samples.append(file_sample)
    
    print("Benign selected:", len(benign_sample))
    print("Attack selected:", len(attack_sample))


Processing: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 25000

Processing: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 25000

Processing: Friday-WorkingHours-Morning.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 1948

Processing: Monday-WorkingHours.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 0

Processing: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 36

Processing: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 2143

Processing: Tuesday-WorkingHours.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 9150

Processing: Wednesday-workingHours.pcap_ISCX.csv
Benign selected: 25000
Attack selected: 25000


In [5]:
ml_data = pd.concat(
    all_samples,
    ignore_index=True
)

print("ML dataset shape:", ml_data.shape)

ML dataset shape: (288277, 79)


In [6]:
ml_data["Binary_Label"] = np.where(
    ml_data["Label"] == "BENIGN",
    0,
    1
)

print(ml_data["Binary_Label"].value_counts())

Binary_Label
0    200000
1     88277
Name: count, dtype: int64


In [7]:
print(
    ml_data["Binary_Label"]
    .value_counts()
    .rename(index={
        0: "BENIGN",
        1: "ATTACK"
    })
)

Binary_Label
BENIGN    200000
ATTACK     88277
Name: count, dtype: int64


In [8]:
X = ml_data.drop(
    columns=columns_to_drop + ["Binary_Label"],
    errors="ignore"
)

y = ml_data["Binary_Label"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (288277, 78)
Target: (288277,)


In [9]:
non_numeric = X.select_dtypes(
    exclude=np.number
).columns

print("Non-numeric features:")
print(non_numeric.tolist())

Non-numeric features:
[]


In [10]:
X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

print("Missing values before:", X.isnull().sum().sum())

Missing values before: 0


In [11]:
valid_rows = X.notnull().all(axis=1)

X = X.loc[valid_rows]
y = y.loc[valid_rows]

print("Final X shape:", X.shape)
print("Final y shape:", y.shape)

print("Missing values:", X.isnull().sum().sum())
print("Infinite values:", np.isinf(X).sum().sum())

Final X shape: (288277, 78)
Final y shape: (288277,)
Missing values: 0
Infinite values: 0


In [12]:
output_path = "../data/CICIDS2017/ml_ready"

os.makedirs(output_path, exist_ok=True)

In [ ]:
final_data = X.copy()
final_data["Binary_Label"] = y

output_file = os.path.join(
    output_path,
    "cicids2017_binary_ml_ready.csv"
)

final_data.to_csv(
    output_file,
    index=False
)

print("ML-ready dataset saved!")
print(output_file)

In [ ]:
print("Saved dataset shape:", final_data.shape)

print("\nLabel distribution:")
print(
    final_data["Binary_Label"]
    .value_counts()
)